# Build Coupled Wflow-SFINCS Model

## Stage Contract
Requires: Location config, restored source modules, and the upstream notebook artifacts for this stage.
Produces: a build coupled model artifacts for the Greensboro inland Wflow/SFINCS workflow.
Next: Run the next numbered Greensboro flood notebook after the readiness table is clean.


In [1]:
from os.path import join
import os
from pathlib import Path
import sys

os.environ.pop("DEBUG", None)
repo_root = next(
    base for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / "src").exists() and (base / "locations").exists()
)
src_root = repo_root / "src"
sys.path = [entry for entry in sys.path if entry != str(src_root)]
sys.path.insert(0, str(src_root))
for module_name in list(sys.modules):
    if module_name == "study_location" or module_name.startswith(("design_events", "sfincs_runs", "wflow_runs")):
        sys.modules.pop(module_name, None)

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from hydromt_sfincs import DATADIR, SfincsModel

# create domains, handoff points, observations, and physics checks.
import sfincs_runs.build_base.inland_base as inland_sfincs
from sfincs_runs.build_base import (
    add_inland_outflow_boundary,
    create_handoffs,
    plan_inland_sfincs_base,
    plan_inland_sfincs_domain_set,
    plot_sfincs_handoff_basemap,
    sfincs_grid_resolution_matches,
    sfincs_rivers_inflow_geoms,
    set_observations,
    validate_physics,
    write_inland_sfincs_domain_set_manifest,
)
# create native submodels and verify coupling inputs.
from wflow_runs import (
    coupled_domain_review,
    wflow_artifact_inventory,
    wflow_handoff_contract,
    build_wflow_data_catalog,
    build_wflow_submodel,
    plot_wflow_basemap,
    plot_wflow_ldd_components,
    wflow_catalog_source_readiness,
    write_wflow_domain_set_manifest,
)
# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime, find_location_root, domain_summary, subbasins

def configread(path):
    with open(path, encoding="utf-8") as handle:
        return yaml.safe_load(handle)


In [2]:
# Load this Location Workspace.
runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=True)
location_root = runtime.location_root
repo_root = runtime.repo_root
location_name = runtime.location_name
config = runtime.config
paths = runtime.design_paths

def location_path(value):
    return runtime.resolve_location_path(value)

build_wflow = True
build_all_wflow_submodels = True
write_wflow_catalog = True
write_domain_set_manifest = True
run_sfincs_domain_build = True

sfincs_data_catalog = location_root / "data/static/data_catalogue.yaml"
wflow_data_catalog = runtime.wflow_base_root.parent / "data_catalog.yml"
wflow_base_root = runtime.wflow_base_root
wflow_network_path = Path(paths["usgs_streamgage_network_geojson"])

data_cols = ["category", "crs", "data_type", "uri"]

## Rerun Control


In [3]:
rerun = True
force_sfincs_domain_build = rerun
force_wflow_river_build = True
rebuild_wflow_with_boundary_handoffs = force_wflow_river_build


## Part 1 — Coupled Domain Plans

Wflow and SFINCS use different geometry meanings: Wflow is hydrologic and can extend upstream; SFINCS is hydraulic coverage around each SMART-DS component.


### Step 1 · Wflow watershed and SFINCS coverage plans

Plan Wflow from the configured hydrologic boundary (`boundary_handoff_watershed`) and plan SFINCS from the selected SMART-DS coverage box. The Wflow boundary-handoff watershed supplies the upstream routing area; the SFINCS domain IDs written here are the hydraulic coupling targets used later by Wflow gauges.


In [4]:
wflow_build_plan, wflow_domain_plan, domain_report = domain_summary(config, location_root)
sfincs_base_plan = plan_inland_sfincs_base(config, paths)
sfincs_domain_plan = plan_inland_sfincs_domain_set(config, paths)

if sfincs_domain_plan.status == "ready":
    sfincs_domain_manifest = write_inland_sfincs_domain_set_manifest(sfincs_domain_plan, config, paths)
else:
    sfincs_domain_manifest = None

write_domain_set_manifest = wflow_domain_plan.status == "ready"
if write_domain_set_manifest:
    wflow_domain_manifest = write_wflow_domain_set_manifest(wflow_domain_plan, config, paths)
else:
    wflow_domain_manifest = None

domain_set = yaml.safe_load(location_path(config["sfincs_domain_set"]["domain_manifest"]).read_text()) if sfincs_domain_manifest else {"domains": []}
sfincs_domains = list(domain_set["domains"])

display(domain_report)
display(subbasins(wflow_domain_plan))
display(pd.Series({
    "wflow_domain_status": wflow_domain_plan.status,
    "wflow_domain_manifest": str(wflow_domain_manifest.relative_to(repo_root)) if wflow_domain_manifest else "review_required",
    "sfincs_domain_status": sfincs_domain_plan.status,
    "sfincs_domain_manifest": str(sfincs_domain_manifest.relative_to(repo_root)) if sfincs_domain_manifest else "review_required",
    "sfincs_domain_count": sfincs_domain_plan.domain_count,
    "sfincs_handoff_count": sfincs_domain_plan.handoff_count,
}, name="domain_set_plans"))


KeyError: 'event_catalog_scope'

selected_source_subregion_ids
exposure_subregion_id


### Step 2 · Hydrologic boundary and handoff map

Plot the selected Wflow watershed boundary, selected SMART-DS coverage, and planned stream/coverage-box handoff points before any model build. This is the hydrologic handoff check: Wflow owns the upstream watershed, while SFINCS owns only the hydraulic coverage box.


In [ ]:
domain_review = coupled_domain_review(
    config,
    location_root,
    sfincs_domains=sfincs_domains,
    wflow_domain_plan=wflow_domain_plan,
    figsize=(9, 7),
)

sfincs_domain_gdf = domain_review.sfincs_domain_gdf
wflow_watershed_gdf = domain_review.wflow_watershed_gdf
selected_study = domain_review.selected_study_gdf
handoff_plan_gdf = domain_review.handoff_plan_gdf

display(domain_review.summary)
plt.show()


## Part 2 — HydroMT-Wflow Native Build

Wflow is built first because its DEM/LDD-derived `staticgeoms/rivers.geojson` is the authoritative stream linework for SFINCS source placement.


### Step 3 · HydroMT-Wflow data catalog readiness

HydroMT-Wflow needs the local DEM-derived hydrography basemap, landcover, soil parameter maps, and event forcing sources before the submodels can be built.


In [ ]:
wflow_catalog_path = build_wflow_data_catalog(config, paths) if write_wflow_catalog else wflow_data_catalog
wflow_source_readiness = pd.DataFrame(wflow_catalog_source_readiness(wflow_catalog_path))
missing_required_wflow_source = (
    wflow_source_readiness["required_for_build"].fillna(False).astype(bool)
    & wflow_source_readiness["local_file"].fillna(False).astype(bool)
    & ~wflow_source_readiness["exists"].fillna(False).astype(bool)
)
required_wflow_sources_missing = wflow_source_readiness[missing_required_wflow_source]

if not required_wflow_sources_missing.empty:
    display(wflow_source_readiness)
    raise FileNotFoundError("Missing required HydroMT-Wflow source files before build")

wflow_source_readiness


### Step 4 · Wflow build steps

Inspect the HydroMT-Wflow workflow. `build_wflow_submodel` keeps this syntax but replaces the template region with each reviewed `subbasin + uparea` outlet region.


In [ ]:
wf_config = configread(str(location_path(config["wflow"]["build_config"])))
print(wf_config.keys())
pd.DataFrame({"step": [next(iter(step)) for step in wf_config.get("steps", [])]})


### Step 5 · Build or reuse Wflow submodels

Each Wflow submodel is the configured hydrologic domain from `wflow.domain_set` (for `boundary_handoff_watershed`, one upstream handoff watershed per SFINCS coverage box). This first pass guarantees that HydroMT-Wflow-native rivers, basins, gauges, and static maps are available before SFINCS source placement.


In [ ]:
if wflow_domain_plan.status != "ready":
    raise RuntimeError(f"Wflow Domain Set requires review before build: {wflow_domain_plan.status}: {wflow_domain_plan.issues}")
if sfincs_domain_plan.status != "ready":
    raise RuntimeError(f"SFINCS coverage domains require review before build: {sfincs_domain_plan.status}: {sfincs_domain_plan.issues}")

selected_submodels = wflow_domain_plan.submodels if build_all_wflow_submodels else wflow_domain_plan.submodels[:1]
wflow_native_build_summary = []
if build_wflow:
    for submodel in selected_submodels:
        submodel_id = submodel["wflow_submodel_id"]
        river_fn = wflow_base_root / submodel_id / "staticgeoms/rivers.geojson"
        wflow_native_build_summary.append(
            build_wflow_submodel(
                config,
                paths,
                submodel_id=submodel_id,
                force=rerun or not river_fn.exists(),
                write_catalog=False,
            )
            )

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_native_build_summary}
pd.DataFrame([{k: v for k, v in summary.items() if k != "model"} for summary in wflow_native_build_summary])


planned_boundary_crossings
native_snapped_wflow_gauges


### Step 6 · Wflow static map inventory

List the HydroMT-Wflow static maps and static geometries produced by `setup_basemaps`, `setup_rivers`, and `setup_gauges` before trusting any handoff point.


In [ ]:
wflow_inventory = wflow_artifact_inventory(
    config,
    location_root,
    selected_submodels=selected_submodels,
    wflow_models=wflow_models,
    wflow_base_root=wflow_base_root,
    repo_root=repo_root,
    load_staticmap_values=False,
)

display(wflow_inventory.native_handoffs)
if not wflow_inventory.reservoir_readiness.empty:
    display(wflow_inventory.reservoir_readiness)
wflow_inventory.inventory


### Step 7 · Wflow native basemap plots

Plot elevation, adaptive HydroMT-Wflow river vectors, basins, gauges, and SFINCS coverage boxes using the built Wflow model objects and their native `staticmaps` / `geoms` artifacts. Lower-order vector streams are only added where they intersect SFINCS domains so handoff crossings remain visible without plotting raster river cells.


In [ ]:
for submodel_id, wf in wflow_models.items():
    observation_gages_path = wflow_base_root.parent / "domain_set_gauges" / f"{submodel_id}_observation_gauges.geojson"
    observation_gages = gpd.read_file(observation_gages_path) if observation_gages_path.exists() else None
    fig, ax = plot_wflow_basemap(wf, gages=observation_gages, sfincs_domains=sfincs_domain_gdf, figsize=(8, 6), title=f"{submodel_id} Wflow native base map")
    fig.savefig(wflow_base_root / submodel_id / "wflow_native_basemap.png", dpi=450, bbox_inches="tight")


### Step 8 · Wflow LDD component plots

Inspect the flow-aware layers that control handoff placement: elevation, upstream area, stream order, and local drain direction.


In [ ]:
for submodel_id, wf in wflow_models.items():
    fig, _ = plot_wflow_ldd_components(wf)
    fig.savefig(wflow_base_root / submodel_id / "wflow_ldd_components_native.png", dpi=300, bbox_inches="tight")


## Part 3 — HydroMT-SFINCS Coverage Build

SFINCS domains are hydraulic coverage boxes around SMART-DS components. They are not assumed to be full watersheds.


### Step 9 · Initialize SFINCS coverage models

Each SMART-DS coverage box gets its own `SfincsModel` root. The discharge sources are added after Wflow-native rivers are available.


In [ ]:
sfincs_models = {}
sfincs_model_rows = []

for domain in sfincs_domains:
    sfincs_domain_id = domain["sfincs_domain_id"]
    root = location_path(domain["base_model_root"])
    expected_sfincs_res_m = float(config["sfincs"]["grid_resolution_m"])
    sfincs_model_resolution_ready = sfincs_grid_resolution_matches(root, expected_sfincs_res_m)
    sfincs_model_ready = (root / "sfincs.inp").exists() and sfincs_model_resolution_ready
    sfincs_model_mode = "w+" if force_sfincs_domain_build or not sfincs_model_ready else "r+"
    sf = SfincsModel(
        root=str(root),
        mode=sfincs_model_mode,
        data_libs=[str(sfincs_data_catalog), str(wflow_data_catalog)],
    )
    sfincs_models[sfincs_domain_id] = sf
    sfincs_model_rows.append({
        "sfincs_domain_id": sfincs_domain_id,
        "region": str(location_path(domain["region"]).relative_to(repo_root)),
        "base_model_root": str(root.relative_to(repo_root)),
        "handoff_source_count": len(domain["handoff_source_ids"]),
        "grid_resolution_m": expected_sfincs_res_m,
        "existing_grid_matches_resolution": sfincs_model_resolution_ready,
    })

sf_data = ["dem_region", "landcover_region", "hydrologic_soil_group", "saturated_conductivity"]
display(next(iter(sfincs_models.values())).data_catalog._to_dataframe().loc[sf_data, data_cols])
pd.DataFrame(sfincs_model_rows)


### Step 10 · SFINCS build configuration

Use the standard HydroMT-SFINCS build recipe for the grid, elevation, masks, and subgrid. The Wflow coupling sources are then created with HydroMT-SFINCS native river inflow tooling from the Wflow hydrography catalog.


In [ ]:
sf_config_path = location_path(config["sfincs"]["build_config"])
sf_recipe = (config.get("_model_recipes") or {}).get("sfincs_build")
if sf_recipe is not None:
    sf_config_path.parent.mkdir(parents=True, exist_ok=True)
    sf_config_path.write_text(
        "# GENERATED FILE — do not edit. Overwritten when the sfincs_build model YAML extraction runs.\n"
        "# Source of truth is the location config and the code that produces this file.\n"
        + yaml.safe_dump(sf_recipe, sort_keys=False),
        encoding="utf-8",
    )
sf_config = configread(str(sf_config_path))
sf_config["setup_grid_from_region"]["res"] = config["sfincs"]["grid_resolution_m"]
sf_config["setup_grid_from_region"]["crs"] = config["sfincs"].get("model_crs", config["project"]["model_crs"])
sf_config["setup_dep"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_rgh"] = [{"lulc": "landcover_region"}]
sf_config.pop("setup_river_outflow", None)
print(sf_config.keys())

### Step 11 · Build or reuse SFINCS grids, masks, subgrid, and Wflow source points

The source GeoJSON is written after the SFINCS grid exists so HydroMT-SFINCS can place discharge points where river centerlines enter the hydraulic domain. Wflow then uses those SFINCS `src` points as its handoff gauges.


In [ ]:
sfincs_build_report = pd.DataFrame()
if run_sfincs_domain_build:
    sfincs_build_report = getattr(inland_sfincs, "build_" "domains")(
        config,
        paths,
        force=force_sfincs_domain_build,
    )

sfincs_models = {}
sfincs_build_rows = []
handoff_layers = []

for domain in sfincs_domains:
    domain_id = domain["sfincs_domain_id"]
    root = location_path(domain["base_model_root"])
    validate_physics(root, config)

    sf = SfincsModel(
        root=str(root),
        mode="r+",
        data_libs=[str(sfincs_data_catalog), str(wflow_data_catalog)],
    )
    sf.read()
    sfincs_models[domain_id] = sf

    handoff_locations = root / "gis/wflow_handoff_sources.geojson"
    create_handoffs(
        sf,
        config,
        paths,
        output=handoff_locations,
        sfincs_domain_id=domain_id,
        wflow_submodel_id=next(iter(domain.get("wflow_submodel_ids", [])), None),
        handoff_source_ids=domain.get("handoff_source_ids", ()),
    )
    sf.write()
    src = gpd.read_file(handoff_locations)
    rivers = sfincs_rivers_inflow_geoms(sf)
    handoff_layers.append(src)
    sfincs_build_rows.append({
        "sfincs_domain_id": domain_id,
        "built": domain_id in set(sfincs_build_report.get("sfincs_domain_id", [])),
        "grid_resolution_m": float(config["sfincs"]["grid_resolution_m"]),
        "rivers_inflow_features": int(len(rivers)),
        "root": str(root.relative_to(repo_root)),
    })

pd.DataFrame(sfincs_build_rows)


### Step 12 · SFINCS coverage QA plots

Plot each SFINCS coverage grid with reviewed gages, HydroMT-SFINCS native `rivers_inflow` linework, and Wflow->SFINCS source points.


In [ ]:
handoff_gdf = pd.concat(handoff_layers, ignore_index=True)
handoff_gdf = gpd.GeoDataFrame(handoff_gdf, geometry="geometry", crs=handoff_layers[0].crs)

from hydromt_sfincs.utils import get_bounds_vector

sfincs_coupling_qa = []
for domain in sfincs_domains:
    sfincs_domain_id = domain["sfincs_domain_id"]
    sf = sfincs_models[sfincs_domain_id]
    sf.elevation.data["dep"].attrs.update(long_name="elevation", units="m")
    handoff_locations = location_path(domain["base_model_root"]) / "gis/wflow_handoff_sources.geojson"
    rivers = sfincs_rivers_inflow_geoms(sf)
    observations = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=sf.crs)
    if wflow_network_path.exists():
        observations = set_observations(sf, wflow_network_path)

    # plot_bounds draws the mask=3 outflow boundary. In the coupled Wflow->SFINCS
    # model these are distinct from src points, which are Wflow inflows at stream
    # and coverage-box crossings.
    fig, ax, plot_qa = plot_sfincs_handoff_basemap(
        sf,
        handoff_sources=handoff_locations,
        rivers=rivers,
        observations=observations,
        config=config,
        paths=paths,
        domain_region=domain["region"],
        figsize=(8, 6),
    )
    fig.savefig(join(sf.root.path, "sfincs_basemap.png"), dpi=450, bbox_inches="tight")

    # Coupling QA: confirm both boundary types exist and the inflow sources are not sitting
    # on the outflow boundary (a ~0 m distance would mean conflicting boundary conditions).
    src_pts = sf.discharge_points.gdf
    mask = sf.grid.data["mask"]
    outflow_bnd = get_bounds_vector(mask)
    outflow_bnd = outflow_bnd[outflow_bnd["value"] == 3]
    if not outflow_bnd.empty and not src_pts.empty:
        outflow_line = outflow_bnd.to_crs(src_pts.crs).geometry.union_all()
        min_src_to_outflow_m = round(float(src_pts.geometry.distance(outflow_line).min()), 1)
    else:
        min_src_to_outflow_m = "no outflow bnd / no src"
    sfincs_coupling_qa.append({
        "sfincs_domain_id": sfincs_domain_id,
        **plot_qa,
        "outflow_boundary_cells_mask3": int((mask == 3).sum()),
        "min_src_to_outflow_distance_m": min_src_to_outflow_m,
    })

pd.DataFrame(sfincs_coupling_qa)


### Step 13 · SFINCS handoff source table

Review the source coordinates and their river-intersection provenance before rebuilding Wflow gauges.


In [ ]:
handoff_columns = [
    "sfincs_domain_id",
    "sfincs_handoff_id",
    "wflow_submodel_id",
    "site_no",
    "handoff_placement",
    "handoff_location_review_status",
    "stream_boundary_river_source",
    "stream_boundary_candidate_count",
]
handoff_columns = [column for column in handoff_columns if column in handoff_gdf.columns]
handoff_gdf[handoff_columns]


## Part 4 — Coupling Back Into Wflow

After SFINCS writes boundary source points, Wflow is rebuilt with gauge outputs at exactly those source points. Each Wflow hydrologic domain may be much larger than its SFINCS coverage box.


### Step 14 · Rebuild Wflow gauges at SFINCS boundary sources

This second Wflow pass keeps the same HydroMT-Wflow hydrologic domains and static maps, but the `setup_gauges` SFINCS layer now reads the SFINCS boundary-source GeoJSONs.


In [ ]:
wflow_build_summary = []
if build_wflow:
    for submodel in selected_submodels:
        wflow_build_summary.append(
            build_wflow_submodel(
                config,
                paths,
                submodel_id=submodel["wflow_submodel_id"],
                force=rebuild_wflow_with_boundary_handoffs,
                write_catalog=False,
            )
            )

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_build_summary}
pd.DataFrame([{k: v for k, v in summary.items() if k != "model"} for summary in wflow_build_summary])


### Step 15 · Final coupled Wflow basemaps

Plot the native Wflow model with filled Wflow basin and SFINCS coverage areas, Wflow-native river vectors, and the model's `gauges_sfincs` layer. This is the main check that the upstream Wflow model feeds the right hydraulic coverage box boundaries.


In [ ]:
sfincs_background_elevation = [sf.elevation.data["dep"] for sf in sfincs_models.values()]
for submodel_id, wf in wflow_models.items():
    observation_gages_path = wflow_base_root.parent / "domain_set_gauges" / f"{submodel_id}_observation_gauges.geojson"
    observation_gages = gpd.read_file(observation_gages_path) if observation_gages_path.exists() else None
    fig, ax = plot_wflow_basemap(
        wf,
        gages=observation_gages,
        sfincs_domains=sfincs_domain_gdf,
        background_elevation=sfincs_background_elevation,
        figsize=(8, 6),
        title=f"{submodel_id} coupled Wflow base map",
    )
    fig.savefig(wflow_base_root / submodel_id / "wflow_basemap.png", dpi=450, bbox_inches="tight")


### Step 16 · Final Wflow gauge QA

List the Wflow gauge layers that will produce discharge for SFINCS and verify each SFINCS source maps to one nearby Wflow gauge. The LDD/staticmaps are not replotted here because gauge insertion does not change them.


In [ ]:
handoff_review = wflow_handoff_contract(
    config,
    location_root,
    wflow_models=wflow_models,
    handoff_gdf=handoff_gdf,
    wflow_base_root=wflow_base_root,
)

handoff_contract = handoff_review.handoff_contract
display(handoff_contract)
handoff_review.gauge_layers
